In [12]:
import difflib
from PIL import Image, ImageDraw, ImageFont
import imageio
import numpy as np

# --- Configuration parameters ---
img_width, img_height = 800, 600
bg_color = "white"
text_color = "black"
highlight_color = "yellow"
score_color = "blue"
margin = 20
font_size = 24
score_font_size = 32
line_spacing = 6  # extra pixels between lines

# Frame durations (in seconds)
frame_duration = 200       # for intermediate frames
freeze_duration = 700      # for the first and last frames

# Load fonts (adjust the path if needed)
try:
    font = ImageFont.truetype("arial.ttf", font_size)
    score_font = ImageFont.truetype("arial.ttf", score_font_size)
except IOError:
    font = ImageFont.load_default()
    score_font = ImageFont.load_default()

# --- Sample paragraphs ---
paragraphs = [
    "Artificial intelligence has revolutionized technology in many fields.",
    "Artificial intelligence has significantly revolutionized modern technology in many fields.",
    "AI has significantly revolutionized modern technology in numerous fields.",
    "AI has truly revolutionized modern technology across numerous fields.",
    "AI has genuinely transformed modern technology across numerous fields.",
    "AI has truly transformed modern technology in many diverse fields.",
    "AI has truly transformed modern technology in many diverse areas.",
    "AI has genuinely transformed technology in many diverse areas.",
    "AI has truly transformed technology in many diverse areas, showing remarkable progress.",
    "AI has truly transformed technology in many diverse areas, showing remarkable progress and sparking innovation.",
    "AI has truly transformed technology across diverse fields, showing remarkable progress and sparking significant innovation."
]
num_frames = len(paragraphs)

# --- Helper function: measure text size using textbbox ---
def get_text_size(draw, text, font):
    bbox = draw.textbbox((0, 0), text, font=font)
    width = bbox[2] - bbox[0]
    height = bbox[3] - bbox[1]
    return width, height

# --- Helper function: draw text with inline highlights ---
def draw_text_with_highlights(draw, text, highlight_word_indices, font, start_x, start_y, max_width):
    """
    Draws text word by word with wrapping.
    Highlights words whose indices are in highlight_word_indices.
    """
    words = text.split()
    space_width, _ = get_text_size(draw, " ", font)
    # Estimate line height (using a sample character)
    _, word_height = get_text_size(draw, "A", font)
    line_height = word_height + line_spacing

    x, y = start_x, start_y
    for idx, word in enumerate(words):
        word_width, word_height = get_text_size(draw, word, font)
        # Wrap to next line if word does not fit
        if x + word_width > start_x + max_width:
            x = start_x
            y += line_height
        # Draw highlight for new words
        if idx in highlight_word_indices:
            pad = 2
            draw.rectangle((x - pad, y - pad, x + word_width + pad, y + word_height + pad), fill=highlight_color)
        draw.text((x, y), word, font=font, fill=text_color)
        x += word_width + space_width
    return y + line_height

# --- Function to compute indices of words added/replaced compared to previous paragraph ---
def compute_highlight_indices(prev_text, curr_text):
    prev_words = prev_text.split()
    curr_words = curr_text.split()
    s = difflib.SequenceMatcher(None, prev_words, curr_words)
    highlight_indices = set()
    for tag, i1, i2, j1, j2 in s.get_opcodes():
        if tag in ("insert", "replace"):
            highlight_indices.update(range(j1, j2))
    return highlight_indices

# --- Generate frames and durations ---
frames = []
durations = []
for i in range(num_frames):
    # Create a new image for this frame
    img = Image.new("RGB", (img_width, img_height), color=bg_color)
    draw = ImageDraw.Draw(img)
    
    # Compute detectability score (linearly decreasing from 100% to 0%)
    detectability = max(0, 100 - int(100 * (i / (num_frames - 1))))
    score_text = f"Detectability: {detectability}%"
    
    # Compute size of score text and center it horizontally
    score_text_width, score_text_height = get_text_size(draw, score_text, score_font)
    score_x = (img_width - score_text_width) // 2
    score_y = margin
    draw.text((score_x, score_y), score_text, font=score_font, fill=score_color)
    
    # Set starting position for paragraph text below the score
    text_start_x = margin
    text_start_y = score_y + score_text_height + margin
    text_max_width = img_width - 2 * margin
    
    # Get the current paragraph and compute differences for highlighting
    curr_text = paragraphs[i]
    if i == 0:
        highlight_indices = set()
    else:
        highlight_indices = compute_highlight_indices(paragraphs[i - 1], curr_text)
    
    # Draw paragraph text with highlights
    draw_text_with_highlights(draw, curr_text, highlight_indices, font, text_start_x, text_start_y, text_max_width)
    
    # Append frame as numpy array
    frames.append(np.array(img))
    
    # Set frame duration: freeze first and last frames
    if i == 0 or i == num_frames - 1:
        durations.append(freeze_duration)
    else:
        durations.append(frame_duration)

# --- Save frames as an animated GIF ---
output_filename = "humanization_animation.gif"
imageio.mimsave(output_filename, frames, duration=durations, loop=0)
print(f"Animation saved as {output_filename}")


Animation saved as humanization_animation.gif
